Dataset: Online Retail II (UCI / Kaggle)
Fuente: Kaggle
Tipo: transacciones de e-commerce

El objetivo es poder limpiar el dataset y solo dejar las ventas validas. Con ventas validas me refiero aquellas que cumplen tener Quantity > 0, Price > 0 y un Customer ID

## Importo el dataset.

In [37]:
import pandas as pd

In [38]:
df = pd.read_csv("online_retail_II.csv")

## Chequeo que se cargo correctamente

In [39]:
df.shape

(1067371, 8)

In [40]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

In [41]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [42]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [43]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


Cuantas filas NO cumplen los requisitos de Venta valida

In [44]:
(df["Quantity"] < 0).sum()


np.int64(22950)

In [45]:
(df["Price"] <= 0).sum()


np.int64(6207)

In [46]:
df["Customer ID"].isnull().sum()


np.int64(243007)

## Excluyo todas aquellas filas que NO cumplan el requisito

In [47]:
df_clean = df[
    (df["Quantity"] > 0) &
    (df["Price"] > 0) &
    (df["Customer ID"].notnull()) ].copy()

## Confirmo que se hayan excluidos esas filas

In [48]:
df.shape

(1067371, 8)

In [49]:
df_clean.shape

(805549, 8)

## Confirmo que todas las filas cumplan los requisitos

In [50]:
(df_clean["Quantity"] < 0).sum()

np.int64(0)

In [51]:
(df_clean["Price"] <= 0).sum()

np.int64(0)

In [52]:
df_clean["Customer ID"].isnull().sum()

np.int64(0)

## Creo columna "Revenue"

In [53]:
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["Price"]


Verifico

In [54]:
df_clean[["Quantity", "Price", "Revenue"]].head()


,Quantity,Price,Revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [55]:
df_clean["Revenue"].min(), df_clean["Revenue"].max()


(np.float64(0.001), np.float64(168469.6))

## Exporto el dataframe limpio

In [56]:
df_clean.to_csv("online_retail_clean.csv", index=False)



Creo que archvio db

In [57]:
import sqlite3
import pandas as pd

In [58]:
conn = sqlite3.connect("retail.db")

paso el dataset clean al archivo

In [59]:
df_clean.to_sql(
    "sales",
    conn,
    if_exists="replace",
    index=False
)


805549

## Verifico

In [60]:
pd.read_sql("SELECT COUNT(*) AS filas FROM sales", conn)


,filas
0,805549


In [61]:
pd.read_sql("SELECT * FROM sales LIMIT 5", conn)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [62]:
pd.read_sql("PRAGMA table_info(sales)", conn)


,cid,name,type,notnull,dflt_value,pk
0,0,Invoice,TEXT,0,None,0
1,1,StockCode,TEXT,0,None,0
2,2,Description,TEXT,0,None,0
3,3,Quantity,INTEGER,0,None,0
4,4,InvoiceDate,TEXT,0,None,0
5,5,Price,REAL,0,None,0
6,6,Customer ID,REAL,0,None,0
7,7,Country,TEXT,0,None,0
8,8,Revenue,REAL,0,None,0


## Métricas Globales del Negocio

En este bloque calculamos las métricas principales del dataset limpio:

- Revenue total
- Cantidad de clientes únicos
- Cantidad de órdenes únicas

Estas métricas permiten entender la magnitud general del negocio antes de filtrar por país o cliente.


In [63]:
query = """
SELECT
    SUM(Revenue) AS total_revenue,
    COUNT(DISTINCT "Customer ID") AS unique_customers,
    COUNT(DISTINCT Invoice) AS unique_orders
FROM sales;
"""

pd.read_sql(query, conn)


,total_revenue,unique_customers,unique_orders
0,1.774343e+07,5878,36969


In [64]:
query_prom = """ SELECT SUM(Revenue) AS total_revenue, COUNT(DISTINCT "Customer ID") AS unique_customers, COUNT(DISTINCT Invoice) AS unique_orders, SUM(Revenue) / COUNT(DISTINCT Invoice) AS ticket_promedio, SUM(Revenue) / COUNT(DISTINCT "Customer ID") AS revenue_promedio_cliente FROM sales"""
pd.read_sql(query_prom,conn)

,total_revenue,unique_customers,unique_orders,ticket_promedio,revenue_promedio_cliente
0,1.774343e+07,5878,36969,479.954264,3018.616737


## Veamoslo por meses

In [65]:
query_por_mes = """ SELECT
strftime('%Y-%m', InvoiceDate) AS anio_mes,
SUM(Revenue) AS total_mes,
COUNT(DISTINCT Invoice) AS ordenes_unicas_mes,
SUM(Revenue) * 1.0 / COUNT(DISTINCT Invoice) AS ticket_promedio_mes,
COUNT(DISTINCT "Customer ID") AS clientes_unicos_mes
FROM sales
GROUP BY anio_mes
ORDER BY anio_mes ASC """
por_mes = pd.read_sql(query_por_mes, conn)

## Lo exporto


In [66]:
por_mes.to_csv("revenue_mensual.csv", index=False)

por_mes.head()

,anio_mes,total_mes,ordenes_unicas_mes,ticket_promedio_mes,clientes_unicos_mes
0,2009-12,686654.160,1512,454.136349,955
1,2010-01,557319.062,1011,551.255254,720
2,2010-02,506371.066,1104,458.669444,772
3,2010-03,699608.991,1524,459.061018,1057
4,2010-04,594609.192,1329,447.410980,942


## Por pais

In [67]:
query_pais = """SELECT
Country,
SUM(Revenue) AS total_pais,
COUNT(DISTINCT Invoice) AS ordenes_unicas_pais,
SUM(Revenue) * 1.0 / COUNT(DISTINCT Invoice) AS ticket_promedio_pais,
COUNT(DISTINCT "Customer ID") AS  clientes_unicos_pais 
FROM sales
GROUP BY Country
ORDER BY total_pais DESC"""

por_pais = pd.read_sql(query_pais, conn)

## Exporto 

In [68]:
por_pais.to_csv("revenue_por_pais.csv", index=False)

por_pais.head()

,Country,total_pais,ordenes_unicas_pais,ticket_promedio_pais,clientes_unicos_pais
0,United Kingdom,1.472315e+07,33541,438.959707,5350
1,EIRE,6.216311e+05,567,1096.351164,5
2,Netherlands,5.542323e+05,228,2430.843596,22
3,Germany,4.312625e+05,789,546.593740,107
4,France,3.552575e+05,614,578.595228,95


## Por mes.

In [69]:
query_pais_mes = """SELECT
strftime('%Y-%m', InvoiceDate) AS anio_mes,
Country,
SUM(Revenue) AS total_pais_mes,
COUNT(DISTINCT Invoice) AS ordenes_unicas_pais_mes,
SUM(Revenue) * 1.0 / COUNT(DISTINCT Invoice) AS ticket_promedio_pais_mes,
COUNT(DISTINCT "Customer ID") AS  clientes_unicos_pais_mes 
FROM sales
GROUP BY anio_mes, Country
ORDER BY anio_mes ASC, Country ASC;"""

por_pais_mes = pd.read_sql(query_pais_mes, conn)

## Exporto

In [70]:
por_pais_mes.to_csv("revenue_pais_mes.csv", index=False)

por_pais_mes.head()

,anio_mes,Country,total_pais_mes,ordenes_unicas_pais_mes,ticket_promedio_pais_mes,clientes_unicos_pais_mes
0,2009-12,Australia,271.10,2,135.550,2
1,2009-12,Austria,1998.34,2,999.170,2
2,2009-12,Belgium,447.60,3,149.200,2
3,2009-12,Channel Islands,989.18,1,989.180,1
4,2009-12,Cyprus,3556.98,4,889.245,3


## Por cliente.

In [71]:
query_cliente = """ SELECT "Customer ID" AS customer_id,
COUNT(DISTINCT Invoice) AS ordenes_cliente,
SUM(Revenue) AS total_cliente,
SUM(Revenue) * 1.0 / COUNT(DISTINCT Invoice) AS ticket_prom_cliente
FROM sales
GROUP BY customer_id
ORDER BY total_cliente DESC"""

pd.read_sql(query_cliente, conn)

,customer_id,ordenes_cliente,total_cliente,ticket_prom_cliente
0,18102.0,145,608821.65,4198.770000
1,14646.0,151,528602.52,3500.678940
2,14156.0,156,313946.37,2012.476731
3,14911.0,398,295972.63,743.649824
4,17450.0,51,246973.09,4842.609608
...,...,...,...,...
5873,15913.0,1,6.30,6.300000
5874,14792.0,1,6.20,6.200000
5875,16738.0,1,3.75,3.750000
5876,13788.0,1,3.75,3.750000


## TOP 20 clientes

In [72]:
query_cliente_top_20 = """ SELECT
    "Customer ID",
    SUM(Revenue) AS revenue_cliente
FROM sales
GROUP BY "Customer ID"
ORDER BY revenue_cliente DESC
LIMIT 20;
"""

top_20_clientes = pd.read_sql(query_cliente_top_20, conn)

## Exporto

In [73]:
top_20_clientes.to_csv("top_20_clientes.csv", index=False)

top_20_clientes

,Customer ID,revenue_cliente
0,18102.0,608821.65
1,14646.0,528602.52
2,14156.0,313946.37
3,14911.0,295972.63
4,17450.0,246973.09
5,13694.0,196482.81
6,17511.0,175603.55
7,16446.0,168472.50
8,16684.0,147142.77
9,12415.0,144458.37


## TOP 20 Productos

In [74]:
query_productos = """ SELECT
StockCode,
Description,
SUM(Quantity) AS unidades_vendidas,
SUM(Revenue) AS revenue_producto,
COUNT(DISTINCT Invoice) AS ordenes_aparicion
FROM sales
GROUP BY StockCode, Description
ORDER BY revenue_producto DESC
LIMIT 20"""

top_20_productos = pd.read_sql(query_productos, conn)

## Exporto

In [75]:
top_20_productos.to_csv("top_20_productos.csv", index=False)

top_20_productos

,StockCode,Description,unidades_vendidas,revenue_producto,ordenes_aparicion
0,22423,REGENCY CAKESTAND 3 TIER,24899,286486.30,3317
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,93640,252072.46,4888
2,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,1
3,M,Manual,9803,152340.57,620
4,85099B,JUMBO BAG RED RETROSPOT,75759,136980.08,2612
5,84879,ASSORTED COLOUR BIRD ORNAMENT,79913,127074.17,2652
6,POST,POSTAGE,5333,126563.04,1803
7,47566,PARTY BUNTING,23607,103880.23,2077
8,23166,MEDIUM CERAMIC TOP STORAGE JAR,77916,81416.73,195
9,22086,PAPER CHAIN KIT 50'S CHRISTMAS,29477,79594.33,1691
